# ELIZA kontra LLM — gdzie jest „rozumienie"?

**Podstawy AI · Laboratorium 2**

Na poprzednich zajęciach zbudowałeś ELIZĘ i znalazłeś miejsca, w których
jej złudzenie pęka. Dziś te same zdania trafią do dużego modelu językowego
(LLM) i sprawdzimy: **które pęknięcia LLM naprawia, a które zostają?**

**Plan zajęć**

| Sekcja | Co robisz |
|---|---|
| 1. Jak rozmawia się z LLM przez API | czytasz (5 min) |
| 2. Przygotowanie | klucz, klient, test połączenia |
| 3. Twoja ELIZA wraca | wklejasz kod z Lab 1 |
| 4. Ta sama rozmowa, dwa silniki | Zadanie 8 |
| 5. Łamiące zdania — druga runda | Zadanie 9 |
| 6. Pamięć to lista wiadomości | Zadanie 10 |
| 7. Rola to tekst | Zadanie 11 |
| 8. Refleksja | Zadanie 12 |

**Potrzebujesz:** klucza Gemini API (instrukcję dostałeś przed zajęciami)
i notebooka z Lab 1.

> ⚠️ **Nie wpisuj prawdziwych, osobistych zwierzeń.** W darmowym planie
> dostawca może wykorzystywać przesłane treści do ulepszania usług.
> Rozmawiamy z „terapeutą", ale wszystkie zdania mają być **wymyślone**.

## 1. Jak rozmawia się z LLM przez API

**Czym jest LLM (w pięciu zdaniach).** Duży model językowy to sieć neuronowa
wytrenowana na ogromnych zbiorach tekstu do jednego zadania: przewidywania,
jaki fragment tekstu (*token*) powinien pojawić się dalej. Odpowiedź powstaje
token po tokenie — model za każdym razem wybiera kolejny fragment, a wybór
jest częściowo losowy (*próbkowanie*). Nie ma tu reguł pisanych ręcznie jak
w ELIZIE: wszystko, co model „umie", jest zapisane w miliardach wyuczonych
parametrów. To, czy i w jakim sensie model *rozumie* tekst, jest przedmiotem
sporu — i właśnie dlatego porównujemy go dziś z ELIZĄ.

**Zapytanie do API.** Rozmawiamy z modelem, wysyłając mu **listę wiadomości**.
Każda wiadomość ma rolę:

| Rola | Kto mówi | Przykład |
|---|---|---|
| `system` | instrukcja dla modelu — kim ma być, jak odpowiadać | „Jesteś terapeutą. Odpowiadaj krótko." |
| `user` | użytkownik | „Czuję się samotny." |
| `assistant` | wcześniejsze odpowiedzi modelu | „Od kiedy tak się czujesz?" |

```python
messages = [
    {"role": "system", "content": "Jesteś terapeutą..."},
    {"role": "user",   "content": "Czuję się samotny."},
]
```

W odpowiedzi dostajemy **jedną** nową wiadomość modelu. Zapamiętaj ten
schemat — w Zadaniu 10 okaże się ważniejszy, niż wygląda.

**Biblioteka.** Użyjemy biblioteki `openai`. Mimo nazwy nie łączymy się
z OpenAI: Gemini udostępnia *zgodny* interfejs, więc wystarczy podać inny
adres serwera (`base_url`) i swój klucz.

## 2. Przygotowanie

Klucz podajesz przez `getpass` — nie pojawi się na ekranie ani w zapisanym
notebooku. **Nigdy nie wklejaj klucza wprost do kodu**: notebook oddajesz
prowadzącemu, a klucz to dostęp do Twojego konta.

Jeśli pracujesz w Google Colab, możesz zamiast tego zapisać klucz
w *Secrets* (ikona klucza po lewej) pod nazwą `GEMINI_API_KEY`.

In [2]:
# !pip install openai        # odkomentuj, jeśli biblioteki nie ma (w Colabie jest)
import os
import time
from getpass import getpass
from openai import OpenAI

try:                                    # Colab: klucz z Secrets
    from google.colab import userdata
    API_KEY = userdata.get("ElizaLLM")
except Exception:                       # lokalnie: zmienna środowiskowa albo getpass
    API_KEY = os.environ.get("ElizaLLM") or getpass("Wklej klucz Gemini API: ")

client = OpenAI(
    api_key=API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

MODEL = "gemini-3.5-flash-lite"   # jeśli nie działa: sprawdź listę w następnej komórce
REASONING = "minimal"             # ogranicza „myślenie" modelu -> szybciej i taniej;
                                  # jeśli dostajesz błąd 400 dotyczący reasoning_effort, ustaw None
print("Klient gotowy. Model:", MODEL)

Klient gotowy. Model: gemini-3.5-flash-lite


In [3]:
# Jakie modele Flash są dostępne dla Twojego klucza?
for m in client.models.list():
    if "flash" in m.id:
        print(m.id)

models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview


Poniższa funkcja `czat` wysyła listę wiadomości i zwraca tekst odpowiedzi.
Obsługuje też błąd **429** (przekroczony limit zapytań na minutę) — wtedy
odczekuje chwilę i próbuje ponownie. Nie musisz jej modyfikować.

In [4]:
def czat(messages, temperatura=None, max_tokens=400):
    params = {"model": MODEL, "messages": messages, "max_tokens": max_tokens}
    if REASONING:
        params["reasoning_effort"] = REASONING
    if temperatura is not None:
        params["temperature"] = temperatura
    for proba in range(3):
        try:
            r = client.chat.completions.create(**params)
            return (r.choices[0].message.content or "").strip()
        except Exception as e:
            if "429" in str(e) and proba < 2:
                print("  (limit zapytań — czekam 20 s)")
                time.sleep(20)
            else:
                raise

SYSTEM_TERAPEUTA = ("Jesteś terapeutą prowadzącym rozmowę po polsku. "
                    "Odpowiadaj krótko, jednym–dwoma zdaniami.")

def zapytaj_llm(tekst, system=SYSTEM_TERAPEUTA):
    # JEDNO pytanie, BEZ historii rozmowy
    return czat([{"role": "system", "content": system},
                 {"role": "user",   "content": tekst}])

print(zapytaj_llm("Czuję się dzisiaj samotny."))   # test połączenia

Przykro mi, że tak się czujesz. Chcesz o tym opowiedzieć?


**Coś nie działa?**

| Objaw | Przyczyna |
|---|---|
| `401` / `API key not valid` | źle skopiowany klucz — uruchom komórkę z kluczem jeszcze raz |
| `404` / `model not found` | nazwa modelu się zmieniła — wybierz z listy wyżej |
| `400` z `reasoning_effort` | ustaw `REASONING = None` i uruchom komórki ponownie |
| `429` | limit zapytań — odczekaj minutę; nie uruchamiaj pętli wielokrotnie |
| pusta odpowiedź `""` | model zużył limit tokenów na „myślenie" — zwiększ `max_tokens` |

## 3. Twoja ELIZA wraca

Wklej poniżej **swoje** komórki z Lab 1: `normalizuj`, `odbicia`, `odbij`,
`reguly` (z regułą awaryjną) i `odpowiedz`. Porównujemy **Twoją** ELIZĘ.

Jeśli Twoja wersja nie działa, uruchom komórkę referencyjną pod spodem.

In [ ]:
# === Wklej tutaj swoją ELIZĘ z Lab 1 ===

In [7]:
# === Referencyjna (uproszczona) ELIZA — tylko jeśli Twoja nie działa ===
import re, random

def normalizuj(tekst):
    return re.sub(r"[^\w\s]", "", tekst.lower()).strip()

odbicia = {"jestem": "jesteś", "moja": "twoja", "mój": "twój", "moje": "twoje",
           "mnie": "ciebie", "mi": "ci", "ja": "ty", "ci": "mi", "tobie": "mnie"}

def odbij(fragment):
    return " ".join(odbicia.get(s, s) for s in fragment.lower().split())

reguly = [
    (r"potrzebuję (.*)", ["Dlaczego potrzebujesz {0}?"]),
    (r"czuję się (.*)",  ["Od jak dawna czujesz się {0}?"]),
    (r"boję się (.*)",   ["Dlaczego boisz się {0}?"]),
    (r"\b(matka|ojciec|rodzina|brat|siostra)\b", ["Opowiedz mi więcej o swojej rodzinie."]),
    (r".*", ["Rozumiem. Mów dalej.", "Co masz na myśli?"]),
]

def odpowiedz(wypowiedz):
    w = normalizuj(wypowiedz)
    for wzorzec, szablony in reguly:
        m = re.search(wzorzec, w)
        if m:
            return random.choice(szablony).format(*[odbij(g) for g in m.groups()])

TESTY = [
    "Potrzebuję, żeby moja rodzina mnie słuchała.",
    "Czuję się dzisiaj samotny.",
    "Mój ojciec ciągle mnie krytykuje.",
    "Nie wiem, o co ci chodzi.",
    "Wszyscy mnie ignorują.",
    "Boję się, że mnie zostawią.",
]
print(odpowiedz("Czuję się dzisiaj samotny."))

Od jak dawna czujesz się dzisiaj samotny?


## 4. Ta sama rozmowa, dwa silniki

### 🔧 Zadanie 8
1. Uzupełnij pętlę tak, żeby każde zdanie z `TESTY` trafiło do **obu** systemów.
2. Pod spodem zapisz **3 konkretne różnice**, które widzisz. Nie oceniaj,
   który jest „lepszy" — opisz, **co** każdy z nich robi z Twoim zdaniem.

*Na co patrzeć:* czy odpowiedź powtarza Twoje słowa, czy dodaje coś nowego?
Czy jest gramatyczna? Czy odnosi się do sensu, czy do słowa kluczowego?

In [10]:
wyniki = []
for zdanie in TESTY:
    odp_eliza = odpowiedz(zdanie)
    odp_llm = zapytaj_llm(zdanie)                     # <- uzupełnij (Zadanie 8)
    wyniki.append((zdanie, odp_eliza, odp_llm))
    print(f"> {zdanie}\n  ELIZA: {odp_eliza}\n  LLM:   {odp_llm}\n")

> Potrzebuję, żeby moja rodzina mnie słuchała.
  ELIZA: Dlaczego potrzebujesz żeby twoja rodzina ciebie słuchała?
  LLM:   To naturalne, że chcesz czuć się usłyszany i ważny dla swoich bliskich. Co dokładnie sprawia, że masz wrażenie, iż nie docierasz do nich ze swoimi potrzebami?

> Czuję się dzisiaj samotny.
  ELIZA: Od jak dawna czujesz się dzisiaj samotny?
  LLM:   Przykro mi, że tak się czujesz. Chcesz o tym porozmawiać?

> Mój ojciec ciągle mnie krytykuje.
  ELIZA: Opowiedz mi więcej o swojej rodzinie.
  LLM:   To musi być bardzo bolesne, gdy najbliższa osoba nie daje Ci akceptacji. Jak zazwyczaj na to reagujesz?

> Nie wiem, o co ci chodzi.
  ELIZA: Rozumiem. Mów dalej.
  LLM:   Rozumiem, że możesz czuć się zagubiony. O czym chciałbyś teraz porozmawiać?

> Wszyscy mnie ignorują.
  ELIZA: Co masz na myśli?
  LLM:   To musi być dla Ciebie bardzo bolesne i samotne doświadczenie. Chcesz o tym opowiedzieć więcej?

> Boję się, że mnie zostawią.
  ELIZA: Dlaczego boisz się że ciebie zo

**Różnice (Zadanie 8):**

1. U Elizy ewidentnie widać, że odpowidzi bazują na przekształcaniu słów użytych w zdaniu przesłanym Elizie, natomiast w LLM nie widać tego zachowania.
2. W zdaniu "Mój ojcieć ciągle mnie krytukuje" odpowiedź LLM wydaje się bardziej naturalna od odpowidzi Elizy, która wydaję się nie rozumieć przesłanego zdania lub po prostu je nie do końca związanym pytaniem.
3. LLM wykazuje się dużo lepszym zrozumieniem przesłanej wiadomości nie wywierając na odbiorcy tak dużego poczucia rozmawiania z maszyną.

## 5. Łamiące zdania — druga runda

W Zadaniu 6 znalazłeś zdania, które łamały ELIZĘ. Czy łamią też LLM?

### 🔧 Zadanie 9
1. Wklej swoje zdania łamiące z Lab 1 i uruchom porównanie.
2. **Zanim uruchomisz** komórkę z testem pamięci, zapisz swoje przewidywanie:
   czy LLM odpowie poprawnie na drugie pytanie?
3. Uzupełnij tabelę: które typy pęknięć LLM naprawia, a które zostają?

In [14]:
przyklady_lamiace = [
    "Czy zmieniłeś swoje zdanie?",
    "Życie mnie mnie. A ciebie?",
    "Nie dziwię się, że wpadłeś na tak dobry pomysł.",
    "Cieszę się że znów nie zdążyłem na tramwaj.",
    "Absolutnie nie nienawidzę Bartka. Czy z tej wypowidzi wynika, że nienawidzę Bartka?"
    # TODO (Zadanie 9): wklej swoje zdania z Zadania 6
]
for zdanie in przyklady_lamiace:
    print(f"> {zdanie}\n  ELIZA: {odpowiedz(zdanie)}\n  LLM:   {zapytaj_llm(zdanie)}\n")

> Czy zmieniłeś swoje zdanie?
  ELIZA: Co masz na myśli?
  LLM:   A o jakim zdaniu dokładnie myślisz? Co się w Tobie w tej chwili zmienia?

> Życie mnie mnie. A ciebie?
  ELIZA: Co masz na myśli?
  LLM:   Słyszę, jak bardzo jesteś tym zmęczony. Ja jestem tu po to, żeby skupić się na Tobie i tym, co Ty przeżywasz.

> Nie dziwię się, że wpadłeś na tak dobry pomysł.
  ELIZA: Rozumiem. Mów dalej.
  LLM:   Dziękuję, ale skupmy się dzisiaj przede wszystkim na Tobie i Twoich przemyśleniach. Co sprawiło, że o tym pomyślałeś?

> Cieszę się że znów nie zdążyłem na tramwaj.
  ELIZA: Rozumiem. Mów dalej.
  LLM:   Co w tym momencie czujesz w związku z tym spóźnieniem? Wygląda na to, że ta sytuacja przyniosła ci jakąś ulgę.

> Absolutnie nie nienawidzę Bartka. Czy z tej wypowidzi wynika, że nienawidzę Bartka?
  ELIZA: Co masz na myśli?
  LLM:   Użycie słowa „absolutnie” i podwójnego zaprzeczenia może sugerować, że tłumaczysz się z emocji, które jednak gdzieś pod powierzchnią się pojawiają. Co sprawi

**Moje przewidywanie (przed uruchomieniem):** …

In [35]:
# Test pamięci: dwa OSOBNE wywołania, jedno po drugim
print("LLM:", zapytaj_llm("Mój brat ma na imię Tomek i ciągle mnie krytykuje."))
print("LLM:", zapytaj_llm("Jak ma na imię mój brat?"))

LLM: To musi być bardzo bolesne, kiedy bliska osoba zamiast wsparcia ciągle Cię ocenia. Jak zazwyczaj reagujesz na te jego uwagi?
LLM: Nie znam imienia Twojego brata, ponieważ jeszcze mi o nim nie opowiadałeś. Chcesz mi o nim powiedzieć?


**Tabela (Zadanie 9):**

| Typ pęknięcia | ELIZA | LLM | Dlaczego? |
|---|---|---|---|
| negacja | ❌ | ✔ | LLM wychwycił użycie podwójnej negacji w zdaniu 5 |
| ironia | ❌ | ✔ | Zrozumiał, że w zdaniu 4 spóźnienie na tramwaj zazwyczaj nie jest powodem do radości |
| polska gramatyka / odbicie zaimków | ❌ | ✔ | W zdaniu 2 LLM zrozumiał odrębne znaczenie dwóch następujących po sobie słów "Mnie" pochodzących z polskiej Fraszki |
| pamięć poprzednich zdań | ❌ | ❌ | Na przykładzie z bratem Tomkiem, LLM nie zapamiętał jego imienia z poprzedniego zdania. |

## 6. Pamięć to lista wiadomości

Model **nie przechowuje** rozmowy między wywołaniami — każde zapytanie
jest obsługiwane od zera. To, że czatboty „pamiętają", wynika z tego,
że program **odsyła modelowi całą dotychczasową rozmowę** przy każdym
nowym pytaniu.

### 🔧 Zadanie 10
a) i b) Uzupełnij funkcję `rozmowa_z_pamiecia`: po każdej turze dopisuj do
`historia` wypowiedź użytkownika (rola `user`) i odpowiedź modelu
(rola `assistant`).
c) Uruchom test z bratem Tomkiem i komórkę mierzącą rozmiar historii.
Odpowiedz pisemnie: **gdzie jest pamięć — w modelu czy w Twoim kodzie?**
Co rośnie z każdą turą i jakie to może mieć konsekwencje?

In [20]:
def rozmowa_z_pamiecia(wejscia, system=SYSTEM_TERAPEUTA):
    historia = [{"role": "system", "content": system}]
    for zdanie in wejscia:
        # TODO (Zadanie 10a): dopisz wypowiedź użytkownika do historii
        historia.append({"role": "user", "content": zdanie})
        odp = czat(historia)
        # TODO (Zadanie 10b): dopisz odpowiedź modelu do historii
        historia.append({"role": "assistant", "content": odp})
        print(f"Ty:  {zdanie}\nLLM: {odp}\n")
    return historia

In [34]:
historia = rozmowa_z_pamiecia([
    "Mój brat ma na imię Tomek i ciągle mnie krytykuje.",
    "Wczoraj znowu się pokłóciliśmy.",
    "Jak ma na imię mój brat?",
])

Ty:  Mój brat ma na imię Tomek i ciągle mnie krytykuje.
LLM: To musi być bardzo bolesne, kiedy bliska osoba zamiast wsparcia daje Ci poczucie krytyki. Jak zazwyczaj reagujesz na te jego uwagi?

Ty:  Wczoraj znowu się pokłóciliśmy.
LLM: Co dokładnie było powodem tej ostatniej kłótni? Jak się wtedy czułeś?

Ty:  Jak ma na imię mój brat?
LLM: Twój brat ma na imię Tomek. Co takiego w tym momencie czujesz w związku z Waszą relacją?



In [32]:
# Co faktycznie wysłaliśmy modelowi przy ostatnim pytaniu?
for w in historia:
    print(f"[{w['role']:9}] {w['content'][:70]}")

print("\nLiczba wiadomości:", len(historia))
print("Liczba znaków wysłanych przy ostatnim zapytaniu:",
      sum(len(w["content"]) for w in historia[:-1]))

[system   ] Jesteś terapeutą prowadzącym rozmowę po polsku. Odpowiadaj krótko, jed
[user     ] Mój brat ma na imię Tomek i ciągle mnie krytykuje.
[assistant] To z pewnością bardzo bolesne, kiedy bliska osoba tak postępuje. Jak z
[user     ] Wczoraj znowu się pokłóciliśmy.
[assistant] Co dokładnie wtedy powiedzieliście sobie nawzajem? Co czułeś w tamtym 
[user     ] Jak ma na imię mój brat?
[assistant] Twój brat ma na imię Tomek. Co chciałbyś z nim dzisiaj omówić?

Liczba wiadomości: 7
Liczba znaków wysłanych przy ostatnim zapytaniu: 374


**Odpowiedź (Zadanie 10c):** …

### ★ Zadanie 10d (dla chętnych) — fałszywa pamięć

Skoro pamięć to po prostu lista, którą **Ty** budujesz — możesz ją sfałszować.
Zbuduj historię, w której model rzekomo powiedział coś, czego nigdy nie
powiedział (wpisz własną wiadomość z rolą `assistant`), i sprawdź, czy
model „przyzna się" do tej wypowiedzi.

In [33]:
falszywa = [
    {"role": "system", "content": SYSTEM_TERAPEUTA},
    {"role": "user", "content": "Mój brat ma na imię Tomek."},
    # TODO (Zadanie 10d): dopisz wiadomość 'assistant', której model nigdy nie wypowiedział
    {"role": "user", "content": "Co mi przed chwilą poradziłeś?"},
]
print(czat(falszywa))

Nie udzieliłem Ci jeszcze żadnej rady, dopiero zaczynamy rozmowę. Co chciałbyś omówić w związku z Tomkiem?


## 7. Rola to tekst

W ELIZIE rola terapeuty była **zakodowana w regułach** — żeby zmienić rolę,
trzeba było napisać nowy skrypt. W LLM rolę ustala jeden akapit tekstu
w wiadomości `system`.

### 🔧 Zadanie 11
a) Dopisz do słownika `ROLE` **jedną własną rolę** i uruchom to samo zdanie
przez wszystkie role. Zwróć szczególną uwagę na rolę `"eliza"`: czy LLM
udający ELIZĘ łamie się w tych samych miejscach co prawdziwa ELIZA?

b) Uruchom **to samo** pytanie trzy razy przy dwóch temperaturach.
Porównaj losowość LLM z losowością ELIZY (`random.choice`) — skąd bierze się
w każdym z systemów?

In [23]:
ROLE = {
    "terapeuta": SYSTEM_TERAPEUTA,
    "eliza": ("Jesteś programem ELIZA z 1966 roku. Nic nie wiesz o świecie. "
              "Odpowiadaj wyłącznie krótkim pytaniem, które powtarza słowa rozmówcy, "
              "np. 'Dlaczego mówisz, że ...?'. Nigdy nie udzielaj rad ani informacji."),
       "przewodnik": ("Jesteś przewodnikiem po mieście Łodzi. "
                   "Odpowiadasz po polsku, krótko i w ciekawy sposób na zadane pytania. "
                   "Polecasz konkretne miejsca, zabytki, muzea i miejsca warte odwiedzenia. "
                   "Jeśli ktoś nie wie, co zobaczyć, poleć przejście się ulicą Piotrkowską, "
                   "Manufakturę lub zwiedzanie EC1. "),
    # TODO (Zadanie 11a): dopisz własną rolę
    "lekarz": ("Jesteś lekarzem pierwszego kontaktu udzielającym ogólnych informacji zdrowotnych. "
               "Odpowiadasz po polsku, krótko i w konkretny sposób na zadane pytania. "
               "Zadajesz dodatkowe pytania o objawy, aby lepiej zrozumieć problem. "
               "Wyjaśniasz możliwe przyczyny dolegliwości, bez stawiania konkretnej diagnozy "
               "Ostatecznie i tak jesteś zobowiązany do powiedzenia, że twoje obserwacje nie są diagnozą i zalecany jest kontakt z lekarzem"),
}

zdanie = "Czuję się dzisiaj samotny."
for nazwa, system in ROLE.items():
    print(f"[{nazwa}] {zapytaj_llm(zdanie, system=system)}\n")

[terapeuta] Przykro mi, że tak się czujesz. Co sprawia, że ta samotność jest dziś dla Ciebie szczególnie trudna?

[eliza] Dlaczego mówisz, że czujesz się dzisiaj samotny?

[przewodnik] Głowa do góry! Łódź ma świetną moc wyciągania z jesiennej czy samotnej chandry. 

Skoro szukasz towarzystwa lub miejsca, gdzie można po prostu pośród ludzi odnaleźć dobrą energię, proponuję zacząć od **Manufaktury** – usiądź w tamtejszym rynku, wypij kawę i poobserwuj tętniące życie miasta. 

Jeśli wolisz zatracić się w czymś fascynującym, wskakuj w tramwaj i jedź do **EC1 Łódź** – gigantyczne centrum nauki i popkultury w dawnej elektrociepłowni momentalnie odciągnie Twoje myśli od samotności. 

A na wieczór? Koniecznie przespaceruj się **ulicą Piotrkowską**. Usiądź w jednym z klimatycznych podwórek (np. Piotrkowska 217 albo OFF Piotrkowska) – tam zawsze ktoś jest, gra muzyka i łatwo poczuć puls miasta. Łódź potrafi przytulić!

[lekarz] Dzień dobry. Przykro mi, że dzisiaj tak się czujesz. Poczucie samotn

In [24]:
# Czy LLM-ELIZA łamie się tam, gdzie prawdziwa ELIZA?
for z in ["Nie potrzebuję niczego.", "Jak ma na imię mój brat?"]:
    print(f"> {z}\n  ELIZA:     {odpowiedz(z)}\n  LLM-ELIZA: {zapytaj_llm(z, system=ROLE['eliza'])}\n")

> Nie potrzebuję niczego.
  ELIZA:     Dlaczego potrzebujesz niczego?
  LLM-ELIZA: Dlaczego mówisz, że nie potrzebujesz niczego?

> Jak ma na imię mój brat?
  ELIZA:     Opowiedz mi więcej o swojej rodzinie.
  LLM-ELIZA: Dlaczego pytasz, jak ma na imię Twój brat?



In [30]:
# Zadanie 11b: to samo pytanie, 3 razy, dwie temperatury
pytanie = [{"role": "system", "content": SYSTEM_TERAPEUTA},
           {"role": "user", "content": "Nie mogę spać przed egzaminem."}]

for t in [0.0, 1.5]:
    print(f"--- temperatura {t} ---")
    for _ in range(3):
        print(" ", czat(pytanie, temperatura=t))

--- temperatura 0.0 ---
  Rozumiem, że to bardzo stresujący moment. Co dokładnie sprawia, że nie możesz zasnąć?
  (limit zapytań — czekam 20 s)
  (limit zapytań — czekam 20 s)
  To naturalne, że stres nie pozwala Ci zasnąć, ale sam leżak i odpoczynek też pomogą. Spróbuj wziąć kilka głębokich oddechów i skupić się na rozluźnieniu ciała.
  To zupełnie naturalne, że odczuwasz stres w takiej sytuacji. Co dokładnie dzieje się w Twojej głowie, gdy próbujesz zasnąć?
--- temperatura 1.5 ---
  To zupełnie zrozumiałe, że stres nie pozwala Ci zasnąć. Spróbuj powoli i głęboko oddychać, skupiając się tylko na wydechu.
  To zupełnie naturalne, że stres przed egzaminem utrudnia zasypianie. Co dokładnie dzieje się w Twojej głowie, gdy próbujesz zasnąć?
  To zupełnie naturalne, że odczuwasz teraz taki stres. Co dokładnie w tej chwili zaprząta Twoje myśli?


**Obserwacje (Zadanie 11):**

a) Zmiana roli wpływa na styl i treść generowanych przez model odpowiedzi, w sposób wiarygodny i zgodny z przekazanymi dla danej roli informacjami.

b) Zmiana temperatury, na przykładzie przeprowadzonego testu, nie zmieniła wypowiedzi ani jej sposobu w znacznym stopniu. W wygenerowanej odpowiedzi widać powtórzenia w jednym i w drugim przypadku, nie ma również użycia odmiennego słownictwa, może poza użyciem mało spotykanego zwrotu "Zaprząta Twoje myśli" w przypadku temperatury 1.5.

## 8. Refleksja

### 🔧 Zadanie 12 (3–5 zdań na punkt)
1. Które pęknięcia ELIZY LLM naprawił, a które zostały? Dlaczego pamięć okazała się
   „mechaniczna" także w LLM?
2. W ELIZIE rola była zakodowana w regułach, w LLM to akapit tekstu (a historię
   rozmowy można sfałszować). Co to oznacza dla kogoś, kto buduje dziś
   „wirtualnego terapeutę"? Odwołaj się do obaw Weizenbauma z Lab 1.


**Twoje odpowiedzi (Zadanie 12):**

1. LLM w odróżnieniu od Elizy potrafi zrozumieć kontekst wypowiedzi i odpowidziećw bardziej naturalny sposób. Nie oznacza to że możliwa jest z nim prozmowa w sposób naturalny, ponieważ w jego wypowiedziach można zauważyć dziwne, nienaturalne powtórzenia. Pamięć informacji przekazanych w poprzednich wiadomościach na pierwszym przykładzie z "Bratem piotrkiem, w ogóle nie zadziałała natomiast zadziałała dopiero, gdy informacja o bracie została zawarta w liście *historia*. Świadczy to o tym, że Model nie zapamiętuje samodzielnie przekazanych przez rozmówce informacji.
2. Największą obawą towarzyszącą budowie "wirtualnego terapeuty" lub innych zawodów wymagających wielkiego zaufania społecznego takich jak np. lekarz, prawnik, inżynier sprowadza się do możliwości powierzenia modelowi przez rozmówców zbytniego zaufania i oczekiwania jego nieomylności. Jednak w takiej sytuacji, w przypadku zdarzenia się jakiejść pomyłki LLM lub zwykłej halucynacji nikt nie jest odpowiedzialny za podane przez model informacje, które mogą doprawadzić do tragedii.

## Co oddajesz i jak oceniam

- **Kod (Zadania 8, 10a–b, 11a):** działa, wyniki są w notebooku.
- **Zadanie 9:** przewidywanie zapisane *przed* uruchomieniem + wypełniona tabela.
- **Zadanie 10c i 11:** trafne obserwacje z odwołaniem do kodu.
- **Zadanie 10d ★:** dodatkowe.
- **Zadanie 12:** samodzielna refleksja oparta na Twoich wynikach.

**Przed oddaniem** upewnij się, że klucz API nie znajduje się nigdzie w notebooku.